# WP4v4 — Notebook 3 : Inférence

**Pipeline** :
```
Image → MAE full ou masque → patch tokens (196 ou 49, 1024)
      → normalisation (cls_norm_stats)
      → f_theta point-wise → tokens espace CLIP (1024)
      → MLP connector LLaVA → tokens espace LLM (4096)
      → LLM (template Vicuna) → description
```
**Normalisation** : une seule statistique (`cls_norm_stats`) utilisee partout.
Les CLS masques et les patch tokens individuels ont des distributions proches
puisque les deux viennent de l'encodeur MAE avec masquage.

## 1. Chargement

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import ViTMAEModel, ViTImageProcessor
from transformers import LlavaForConditionalGeneration, CLIPImageProcessor, LlamaTokenizer
from datasets import load_dataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

class ProjectionMLP(nn.Module):
    def __init__(self, dim=1024, hidden_dim=2048, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, dim),
        )
    def forward(self, x): return F.normalize(self.net(x), dim=-1)

f_theta = ProjectionMLP().to(DEVICE)
f_theta.load_state_dict(torch.load('wp4v4_ftheta_best.pt'))
f_theta.eval()

# Une seule statistique de normalisation
cls_norm = torch.load('wp4v4_cls_norm_stats.pt')
z_mean   = cls_norm['mean'].cpu()
z_std    = cls_norm['std'].cpu()
print('f_theta et stats charges')


In [ ]:
mae_processor = ViTImageProcessor(
    size={'height': 224, 'width': 224},
    image_mean=[0.485, 0.456, 0.406],
    image_std=[0.229, 0.224, 0.225],
)
mae_encoder = ViTMAEModel.from_pretrained('./vit-mae-large').to(DEVICE)
mae_encoder.eval()

llava_full = LlavaForConditionalGeneration.from_pretrained(
    './llava-1.5-7b-hf', torch_dtype=torch.float16
).to(DEVICE)
llava_full.eval()
llm          = llava_full.language_model
vision_tower = llava_full.vision_tower
mlp_conn     = llava_full.multi_modal_projector
clip_proc    = CLIPImageProcessor.from_pretrained('./llava-1.5-7b-hf')
tokenizer    = LlamaTokenizer.from_pretrained('./llava-1.5-7b-hf', use_fast=False)
print(f'VRAM : {torch.cuda.memory_allocated()/1e9:.1f} GB')


## 2. Fonctions utilitaires

In [ ]:
SYSTEM    = ('A chat between a curious user and an artificial intelligence assistant. '
             'The assistant gives helpful, detailed, and polite answers to the user questions.')
USER_TEXT = 'Describe this image in one sentence.'
before_ids    = tokenizer(f'{SYSTEM} USER: ', return_tensors='pt', add_special_tokens=True).input_ids.to(DEVICE)
after_ids     = tokenizer(f'\n{USER_TEXT} ASSISTANT:', return_tensors='pt', add_special_tokens=False).input_ids.to(DEVICE)
before_embeds = llm.get_input_embeddings()(before_ids).half()
after_embeds  = llm.get_input_embeddings()(after_ids).half()


def project_tokens(tokens):
    """
    Projette N tokens MAE via f_theta.
    tokens : Tensor (N, 1024) — CLS ou patch tokens
    Retourne : Tensor (N, 1024) L2-normalise dans l'espace CLIP
    """
    z_norm = (tokens.cpu().float() - z_mean) / z_std
    with torch.no_grad():
        return f_theta(z_norm.to(DEVICE)).cpu().float()


def llm_describe(visual_tokens_clip):
    """tokens CLIP (N, 1024) -> MLP connector -> LLM -> description."""
    with torch.no_grad():
        visual_llm = mlp_conn(visual_tokens_clip.half().to(DEVICE))
    inputs_embeds = torch.cat([before_embeds, visual_llm.unsqueeze(0), after_embeds], dim=1)
    with torch.no_grad():
        out_ids = llm.generate(
            inputs_embeds=inputs_embeds, max_new_tokens=50,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()


def get_mae_tokens(image_pil, seed=None):
    """Retourne CLS (1024,) + patch tokens (196 ou 49, 1024) + visible_ids."""
    inp = mae_processor(images=image_pil, return_tensors='pt')
    inp = {k: v.to(DEVICE) for k, v in inp.items()}
    if seed is None:
        noise = torch.zeros(1, 196).to(DEVICE)
    else:
        gen   = torch.Generator().manual_seed(seed)
        noise = torch.rand(1, 196, generator=gen).to(DEVICE)
    with torch.no_grad():
        out = mae_encoder(**inp, noise=noise)
    hs          = out.last_hidden_state[0]
    cls_token   = hs[0].cpu().float()
    patch_tokens = hs[1:].cpu().float()
    mask        = out.mask[0].cpu()
    visible_ids = torch.where(mask == 0)[0].tolist()
    return cls_token, patch_tokens, visible_ids


def describe_llava_native(image_pil):
    """LLaVA natif : 576 patch tokens CLIP -> MLP connector -> LLM."""
    inp = clip_proc(images=image_pil, return_tensors='pt', do_rescale=True)
    pix = inp['pixel_values'].to(DEVICE).half()
    with torch.no_grad():
        vis     = vision_tower(pix).last_hidden_state[:, 1:]
        vis_llm = mlp_conn(vis)[0]
    inputs_embeds = torch.cat([before_embeds, vis_llm.unsqueeze(0), after_embeds], dim=1)
    with torch.no_grad():
        out_ids = llm.generate(
            inputs_embeds=inputs_embeds, max_new_tokens=50,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()

print('Fonctions definies')


## 3. Vérification discriminabilité après projection

In [ ]:
ds = load_dataset('parquet', data_files={
    'validation': './imagenet100/data/validation-*.parquet',
})

raw_vecs, proj_vecs = [], []
for i in range(5):
    item      = ds['validation'][i]
    image_pil = item['image'].convert('RGB').resize((224, 224))
    # CLS masque (seed=0) — meme distribution que l'entrainement
    cls_token, _, _ = get_mae_tokens(image_pil, seed=0)
    z_proj = project_tokens(cls_token.unsqueeze(0))[0]
    raw_vecs.append(F.normalize(cls_token.unsqueeze(0), dim=-1)[0])
    proj_vecs.append(z_proj)
    print(f'{item["text"][:38]}')

raw_mat  = torch.stack(raw_vecs)
proj_mat = torch.stack(proj_vecs)
print('\nSimilarites CLS MAE masque brut :')
print((raw_mat @ raw_mat.T).numpy().round(4))
print('\nSimilarites apres projection f_theta :')
print((proj_mat @ proj_mat.T).numpy().round(4))


## 4. Sanity check — LLaVA natif vs MAE projeté

In [ ]:
N_IMAGES = 5
print(f'{"Classe":<38} {"LLaVA natif (576 patches)":<45} {"MAE -> f_theta -> MLP (196 patches)"}')
print('-' * 125)

for i in range(N_IMAGES):
    item      = ds['validation'][i]
    image_pil = item['image'].convert('RGB')
    label     = item['text'][:35]

    desc_llava = describe_llava_native(image_pil)

    # MAE : 196 patch tokens -> f_theta -> MLP connector -> LLM
    image_224 = image_pil.resize((224, 224))
    _, patch_tokens, _ = get_mae_tokens(image_224, seed=None)  # full encoding
    clip_space = project_tokens(patch_tokens)                   # (196, 1024)
    desc_mae   = llm_describe(clip_space)

    print(f'{label:<38} {desc_llava:<45} {desc_mae}')


## 5. Test avec masquage

In [ ]:
IMG_IDX = 0; SEED = 42
item      = ds['validation'][IMG_IDX]
image_pil = item['image'].convert('RGB').resize((224, 224))
label_txt = item['text']

_, patch_masked, visible_ids = get_mae_tokens(image_pil, seed=SEED)
_, patch_full,   _           = get_mae_tokens(image_pil, seed=None)

print(f'Classe : {label_txt} | Patches visibles : {len(visible_ids)}')
print(f'Description (49 patches masques) : {llm_describe(project_tokens(patch_masked))}')
print(f'Description (196 patches full)   : {llm_describe(project_tokens(patch_full))}')

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
overlay = np.array(image_pil).copy().astype(float)
for pid in range(196):
    if pid not in visible_ids:
        r, c = pid // 14, pid % 14
        overlay[r*16:(r+1)*16, c*16:(c+1)*16] *= 0.15
ax.imshow(overlay.astype(np.uint8))
ax.set_title(label_txt, fontsize=9); ax.axis('off')
plt.tight_layout()
plt.savefig('wp4v4_image_masque.png', dpi=150, bbox_inches='tight')
plt.show()
